# Week 3: Deploy the categorizer (Tasks 1–3)

You'll build a real FastAPI service in `service/`. Cells that start with `%%writefile` **save the cell into a file** instead of running it.

Plan: `docs/superpowers/plans/2026-10-21-week3-categorizer-deployment.md`

**TDD rhythm for every task:** write the test → watch it fail → write the code → watch it pass.

⚠️ Re-running a `%%writefile` cell overwrites that file. Make changes here, not in the files directly.

## Setup
Install the tools (run once), then load the helpers that run commands inside `service/`.

In [6]:
%pip install -q fastapi httpx pytest uvicorn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: D:\Web Projects\machine-learning\.venv\Scripts\python.exe -m pip install --upgrade pip


In [55]:
import shutil, subprocess, sys
from pathlib import Path

SERVICE = Path('../service').resolve()
(SERVICE / 'tests').mkdir(parents=True, exist_ok=True)

def run(cmd):
    """Run a command inside service/ and show its output."""
    try:
        r = subprocess.run(
            cmd,
            cwd=SERVICE,
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace'
        )
        print(r.stdout + r.stderr)
        print(f'Exit code: {r.returncode}')
    except FileNotFoundError:
        print(f'{cmd[0]} is not installed or not on PATH')

def pytest(*args):
    run([sys.executable, '-m', 'pytest', *args, '-v'])

print('service folder:', SERVICE)

service folder: D:\Web Projects\machine-learning\service


---
# Task 1: Text cleaning module

The API must clean text **exactly** like training did. If it doesn't, accuracy silently drops. This is called *training/serving skew*.

### 1.1 Write the test first

In [19]:
%%writefile ../service/tests/test_clean.py
from clean import clean_description


def test_matches_notebook_cleaning():
    assert clean_description('JOLLIBEE*4821') == 'jollibee'
    assert clean_description('Jollibee Makati') == 'jollibee'
    assert clean_description('  Starbucks ') == 'starbucks'


Overwriting ../service/tests/test_clean.py


### 1.2 Watch it fail
Expected: `ModuleNotFoundError: No module named 'clean'`. That's correct, because the code doesn't exist yet.

In [21]:
pytest('tests/test_clean.py')

============================= test session starts =============================
platform win32 -- Python 3.14.5, pytest-9.1.1, pluggy-1.7.0 -- D:\Web Projects\machine-learning\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: D:\Web Projects\machine-learning\service
plugins: anyio-4.15.1, platformdirs-4.13.5
collecting ... collected 1 item

tests/test_clean.py::test_matches_notebook_cleaning PASSED               [100%]

============================== 1 passed in 0.04s ==============================



### 1.3 Write `clean.py`
Hints:
- Copy `BRANCHES` and `clean_description` from cell **B1** of your Week 2 notebook
- You need `import re`
- Add a one-line comment: why must this stay identical to the notebook?

In [63]:
%%writefile ../service/clean.py
# Use the same cleaning as training so the model receives text in the format it learned.
import re

BRANCHES = [
    'sm north', 'megamall', 'bgc',
    'makati', 'qc', 'cubao', 'ortigas'
]

def clean_description(text):
    text = text.lower()
    text = re.sub(r'\*?\d+', '', text)

    for branch in BRANCHES:
        text = text.replace(branch, '')

    return text.strip()

Overwriting ../service/clean.py


### 1.4 Watch it pass
Expected: `1 passed`

In [62]:
pytest('tests/test_clean.py')

============================= test session starts =============================
platform win32 -- Python 3.14.5, pytest-9.1.1, pluggy-1.7.0 -- D:\Web Projects\machine-learning\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: D:\Web Projects\machine-learning\service
plugins: anyio-4.15.1, platformdirs-4.13.5
collecting ... collected 1 item

tests/test_clean.py::test_matches_notebook_cleaning PASSED               [100%]

============================== 1 passed in 0.04s ==============================

Exit code: 0


---
# Task 2: The `/categorize` API

```
POST /categorize  {"description": "JOLLIBEE*4821", "type": "expense"}
                → {"category": "Food & Drinks", "confidence": 0.91}
                → {"category": null, "confidence": 0.32}   (below 0.5: no suggestion)
GET  /health    → {"status": "ok", "model_version": "2026-10-21"}
```

### 2.1 Release the model
Copy your Week 2 model into the service. This copy **gets committed**, because Vercel builds from GitHub.

In [25]:
(SERVICE / 'model').mkdir(exist_ok=True)
shutil.copy('../models/categorizer.joblib', SERVICE / 'model' / 'categorizer.joblib')
print((SERVICE / 'model' / 'categorizer.joblib').stat().st_size // 1024, 'KB')

139 KB


### 2.2 Write the tests first
Read each test. Together they **are** the spec for your API.

In [33]:
%%writefile ../service/tests/test_app.py
from fastapi.testclient import TestClient

from app import INCOME, app

client = TestClient(app)
APP_DEFAULTS = {'Food & Drinks', 'Transportation', 'Bills & Utilities', 'Shopping', 'Groceries',
                'Coffee & Snacks', 'Healthcare', 'Entertainment', 'Salary', 'Freelance / Client', 'Other Income'}


def post(description, type='expense'):
    return client.post('/categorize', json={'description': description, 'type': type})


def test_jollibee_is_food():
    r = post('Jollibee')
    assert r.status_code == 200
    assert r.json()['category'] == 'Food & Drinks'


def test_payroll_income_is_salary():
    assert post('Payroll', 'income').json()['category'] == 'Salary'


def test_unknown_store_has_no_suggestion():
    assert post('UNKNOWN STORE').json()['category'] is None


def test_expense_never_gets_income_category():
    for text in ['Payroll', 'Upwork', 'Gift', 'Refund']:
        assert post(text).json()['category'] not in INCOME


def test_rejects_empty_and_too_long():
    assert post('   ').status_code == 422
    assert post('x' * 201).status_code == 422
    assert post('x' * 200).status_code == 200


def test_text_that_cleans_to_empty_returns_null():
    for text in ['12345', '*1234', 'makati']:
        r = post(text)
        assert r.status_code == 200
        assert r.json()['category'] is None


def test_non_ascii_text_does_not_crash():
    for text in ['Kape ☕ sa kanto', 'Ñaw']:
        assert post(text).status_code == 200


def test_rejects_unknown_type():
    assert post('Jollibee', 'transfer').status_code == 422


def test_health():
    assert client.get('/health').json() == {'status': 'ok', 'model_version': '2026-10-21'}


def test_model_classes_match_app_defaults():
    from app import model
    assert set(model.classes_) <= APP_DEFAULTS


Overwriting ../service/tests/test_app.py


### 2.3 Watch them fail
Expected: `ModuleNotFoundError: No module named 'app'`

In [45]:
pytest()

============================= test session starts =============================
platform win32 -- Python 3.14.5, pytest-9.1.1, pluggy-1.7.0 -- D:\Web Projects\machine-learning\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: D:\Web Projects\machine-learning\service
plugins: anyio-4.15.1, platformdirs-4.13.5
collecting ... collected 11 items

tests/test_app.py::test_jollibee_is_food FAILED                          [  9%]
tests/test_app.py::test_payroll_income_is_salary FAILED                  [ 18%]
tests/test_app.py::test_unknown_store_has_no_suggestion PASSED           [ 27%]
tests/test_app.py::test_expense_never_gets_income_category PASSED        [ 36%]
tests/test_app.py::test_rejects_empty_and_too_long FAILED                [ 45%]
tests/test_app.py::test_text_that_cleans_to_empty_returns_null PASSED    [ 54%]
tests/test_app.py::test_non_ascii_text_does_not_crash PASSED             [ 63%]
tests/test_app.py::test_rejects_unknown_type PASSED                      [ 72%]
tests/t

### 2.4 Write `app.py`
Fill in each TODO. Hints:
- **Load the model once**, at the top of the file, not inside the endpoint. Loading on every request would be slow.
- **Validation:** with pydantic's `StringConstraints`, FastAPI returns 422 for bad input automatically.
- **Type filter:** `model.predict_proba([cleaned])[0]` gives one probability per entry in `model.classes_`. Set the probabilities of the *wrong* type to 0, divide by the new sum, then take the biggest. Hint: `keep = np.array([(c in INCOME) == (req.type == 'income') for c in model.classes_])`, then `np.where(keep, probs, 0.0)`.
- If the cleaned text is empty, return `category: None` without calling the model.
- Return plain Python types: `float(...)` and `str(...)`. NumPy types can break JSON.

In [46]:
%%writefile ../service/app.py
from pathlib import Path
from typing import Annotated, Literal

import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, StringConstraints

from clean import clean_description

THRESHOLD = 0.5
MODEL_VERSION = '2026-10-21'
INCOME = {'Salary', 'Freelance / Client', 'Other Income'}

# Load the saved pipeline once when this module is imported
model = joblib.load(
    Path(__file__).parent / 'model' / 'categorizer.joblib'
)

app = FastAPI()


class CategorizeRequest(BaseModel):
    description: Annotated[
        str,
        StringConstraints(
            strip_whitespace=True,
            min_length=1,
            max_length=200
        )
    ]
    type: Literal['expense', 'income']


@app.post('/categorize')
def categorize(req: CategorizeRequest):
    cleaned = clean_description(req.description)

    if not cleaned:
        return {'category': None, 'confidence': 0.0}

    probs = model.predict_proba([cleaned])[0]

    # Keep only categories matching the requested transaction type
    keep = np.array([
        (category in INCOME) == (req.type == 'income')
        for category in model.classes_
    ])
    probs = np.where(keep, probs, 0.0)

    total = probs.sum()
    if total == 0:
        return {'category': None, 'confidence': 0.0}

    # Rescale the remaining probabilities so they sum to 1
    probs = probs / total

    best_index = int(np.argmax(probs))
    confidence = float(probs[best_index])

    category = (
        str(model.classes_[best_index])
        if confidence >= THRESHOLD
        else None
    )

    return {'category': category, 'confidence': confidence}


@app.get('/health')
def health():
    return {'status': 'ok', 'model_version': MODEL_VERSION}

Overwriting ../service/app.py


### 2.5 Pin the versions
The model was trained with scikit-learn **1.9.1**. Loading it with another version can break it or quietly change its behavior.

Run the cell below, then write **`service/requirements.txt`** with one `name==version` per line for: fastapi, scikit-learn, joblib, numpy, scipy.
Don't include pytest or httpx; those are only needed for testing.

In [47]:
from importlib.metadata import version
for pkg in ['fastapi', 'scikit-learn', 'joblib', 'numpy', 'scipy']:
    print(f'{pkg}=={version(pkg)}')

fastapi==0.156.4
scikit-learn==1.9.1
joblib==1.6.0
numpy==2.5.4
scipy==1.18.1


In [48]:
%%writefile ../service/requirements.txt
fastapi==0.156.4
scikit-learn==1.9.1
joblib==1.6.0
numpy==2.5.4
scipy==1.18.1

Writing ../service/requirements.txt


### 2.6 Watch them pass
Expected: `11 passed` (1 from Task 1 and 10 from Task 2)

In [49]:
pytest()

============================= test session starts =============================
platform win32 -- Python 3.14.5, pytest-9.1.1, pluggy-1.7.0 -- D:\Web Projects\machine-learning\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: D:\Web Projects\machine-learning\service
plugins: anyio-4.15.1, platformdirs-4.13.5
collecting ... collected 11 items

tests/test_app.py::test_jollibee_is_food PASSED                          [  9%]
tests/test_app.py::test_payroll_income_is_salary PASSED                  [ 18%]
tests/test_app.py::test_unknown_store_has_no_suggestion PASSED           [ 27%]
tests/test_app.py::test_expense_never_gets_income_category PASSED        [ 36%]
tests/test_app.py::test_rejects_empty_and_too_long PASSED                [ 45%]
tests/test_app.py::test_text_that_cleans_to_empty_returns_null PASSED    [ 54%]
tests/test_app.py::test_non_ascii_text_does_not_crash PASSED             [ 63%]
tests/test_app.py::test_rejects_unknown_type PASSED                      [ 72%]
tests/t

### 2.7 Try the real server
This starts the API like production would, sends one request, then stops it.

In [50]:
import time, httpx
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app:app', '--port', '8000'], cwd=SERVICE)
try:
    time.sleep(3)
    for text in ['JOLLIBEE*4821', 'asdfgh']:
        print(text, '→', httpx.post('http://127.0.0.1:8000/categorize', json={'description': text, 'type': 'expense'}).json())
finally:
    server.terminate()

JOLLIBEE*4821 → {'category': 'Food & Drinks', 'confidence': 0.8870059946719897}
asdfgh → {'category': None, 'confidence': 0.33145523532308147}


---
# Task 3: Docker

Docker packages the service, with its exact Python version and libraries, into an **image** that runs the same on any computer.

### 3.1 Is Docker ready?
Expected: both **Client** and **Server** versions. If you see "not installed" or a Server error, install or start **Docker Desktop** first.

In [51]:
run(['docker', 'version'])

Client:
 Version:           29.8.2
 API version:       1.56
 Go version:        go1.26.8
 Git commit:        7fc2dff
 Built:             Wed Sep 30 19:35:42 2026
 OS/Arch:           windows/amd64
 Context:           desktop-linux

Server: Docker Desktop 4.95.0 (242831)
 Engine:
  Version:          29.8.2
  API version:      1.56 (minimum version 1.40)
  Go version:       go1.26.8
  Git commit:       8af9fe3
  Built:            Wed Sep 30 19:32:25 2026
  OS/Arch:          linux/amd64
  Experimental:     false
 containerd:
  Version:          v2.3.6
  GitCommit:        ee2735368117d2eb259779949d5e75cdafec9761
 runc:
  Version:          1.5.1
  GitCommit:        v1.5.1-0-g8f2685a4
 docker-init:
  Version:          0.19.0
  GitCommit:        de40ad0



### 3.2 Write the Dockerfile
Hints, one instruction per line:
1. `FROM python:3.14-slim`: a small Python base image
2. `WORKDIR /app`
3. `COPY requirements.txt .` then `RUN pip install --no-cache-dir -r requirements.txt uvicorn pytest httpx`
   (requirements are copied first so Docker can reuse this slow step when only your code changes)
4. `COPY . .`
5. `CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]`

**Think:** why `0.0.0.0` instead of `127.0.0.1` inside a container?

In [52]:
%%writefile ../service/Dockerfile
FROM python:3.14-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt uvicorn pytest httpx
COPY . .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing ../service/Dockerfile


In [53]:
%%writefile ../service/.dockerignore
__pycache__/
.pytest_cache/
.venv/

Writing ../service/.dockerignore


### 3.3 Build the image
The first build takes a few minutes. Expected at the end: `naming to docker.io/library/spendify-categorizer`

In [56]:
run(['docker', 'build', '-t', 'spendify-categorizer', '.'])

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 251B done
#1 DONE 0.0s

#2 [auth] library/python:pull token for registry-1.docker.io
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.14-slim
#3 DONE 1.4s

#4 [internal] load .dockerignore
#4 transferring context: 78B done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 513B done
#5 DONE 0.0s

#6 [1/5] FROM docker.io/library/python:3.14-slim@sha256:fbecd461fc72d21b2143c6eaba25143738079154a6d0ea3da3aec724f50eed4a
#6 resolve docker.io/library/python:3.14-slim@sha256:fbecd461fc72d21b2143c6eaba25143738079154a6d0ea3da3aec724f50eed4a 0.0s done
#6 DONE 0.0s

#7 [2/5] WORKDIR /app
#7 CACHED

#8 [3/5] COPY requirements.txt .
#8 CACHED

#9 [4/5] RUN pip install --no-cache-dir -r requirements.txt uvicorn pytest httpx
#9 CACHED

#10 [5/5] COPY . .
#10 CACHED

#11 exporting to image
#11 exporting layers done
#1

### 3.4 Run the tests inside the container
Expected: `11 passed`

In [59]:
run(['docker', 'run', '--rm', 'spendify-categorizer', 'python', '-m', 'pytest', '-q'])

...........                                                              [100%]
=============================== warnings summary ===============================
../usr/local/lib/python3.14/site-packages/fastapi/testclient.py:1
  /usr/local/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
11 passed, 1 warning in 1.69s

Exit code: 0


### 3.5 Run the server in the container and call it
Expected: `Food & Drinks`

In [58]:
run(['docker', 'run', '--rm', '-d', '-p', '8000:8000', '--name', 'cat', 'spendify-categorizer'])
time.sleep(4)
try:
    print(httpx.post('http://127.0.0.1:8000/categorize', json={'description': 'JOLLIBEE*4821', 'type': 'expense'}).json())
finally:
    run(['docker', 'stop', 'cat'])

a9786f6c77b50bd03dc09391a903f3eefa26384ba14d52031486a3684e39b321

Exit code: 0
{'category': 'Food & Drinks', 'confidence': 0.8870059946719897}
cat

Exit code: 0


---
## Done with Tasks 1–3?
Tell Claude "Tasks 1–3 done". It will review the files, commit them, and help you deploy to Vercel (Task 4).